# Notebook 3: Empirical Results — Cross-Sectional Return Prediction

This notebook reproduces the main empirical experiment on synthetic data:

1. Feature extraction (signatures vs. hand-crafted benchmarks)
2. Expanding-window OOS prediction
3. OOS R², Diebold-Mariano test
4. Fama-MacBeth regression
5. Portfolio sort analysis

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data.synthetic import generate_gbm_with_leverage
from pathsig.features import SignatureFeatureExtractor, BenchmarkFeatureExtractor
from pathsig.models import CrossSectionalPredictor
from pathsig.evaluation import (
    out_of_sample_r_squared,
    diebold_mariano_test,
    fama_macbeth_regression,
    portfolio_sort_analysis,
    prediction_summary,
)

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11
print('Loaded.')

## 1. Generate Synthetic Panel Data

In [ ]:
print('Generating synthetic panel (Heston model with leverage, ρ=−0.7)…')
panel = generate_gbm_with_leverage(
    n_paths=150, n_steps=504,  # ~2 years of daily data
    leverage_corr=-0.7, seed=42,
)
panel = panel.sort_values(['ticker', 'date']).reset_index(drop=True)
panel['ret_forward'] = panel.groupby('ticker')['log_return'].shift(-1)
panel = panel.dropna(subset=['ret_forward']).reset_index(drop=True)

print(f'Panel: {len(panel):,} rows, {panel["ticker"].nunique()} stocks, {panel["date"].nunique()} dates')

## 2. Extract Features

In [ ]:
sig_extractor = SignatureFeatureExtractor(
    channels=['log_return', 'realized_vol', 'log_volume'],
    window=21, depth=2,
    augmentations=['time', 'lead_lag'],
    normalize=True,
)
bench_extractor = BenchmarkFeatureExtractor(window=21)

print('Extracting signature features…')
sig_features = sig_extractor.fit_transform(panel)
print(f'  → {sig_features.shape}: {sig_features.shape[1]-2} signature features')

print('Extracting benchmark features…')
bench_features = bench_extractor.fit_transform(panel)
print(f'  → {bench_features.shape}: {bench_features.shape[1]-2} benchmark features')

## 3. Expanding-Window OOS Prediction

In [ ]:
def run_oos(features, panel, label, min_train=100):
    id_col = 'ticker'
    feat_cols = [c for c in features.columns if c not in ('date', id_col)]
    merged = features.merge(
        panel[['date', id_col, 'ret_forward']].dropna(),
        on=['date', id_col], how='inner'
    ).dropna(subset=feat_cols + ['ret_forward'])
    
    predictor = CrossSectionalPredictor(model_type='ridge', standardize=True)
    print(f'Running OOS ({label})…', end=' ')
    preds = predictor.expanding_window_predict(
        panel=merged, feature_cols=feat_cols,
        return_col='ret_forward', min_train_periods=min_train,
    )
    summary = prediction_summary(preds)
    print(f'OOS R²={summary["oos_r2"]*100:.3f}%, hit={summary["hit_rate"]:.3f}')
    return preds, summary

pred_sig,   sum_sig   = run_oos(sig_features,   panel, 'signature')
pred_bench, sum_bench = run_oos(bench_features,  panel, 'benchmark')

In [ ]:
# Summary table
results_df = pd.DataFrame([
    {'Model': 'Signature', **sum_sig},
    {'Model': 'Benchmark', **sum_bench},
]).set_index('Model')[['oos_r2', 'hit_rate', 'corr_pearson', 'corr_spearman', 'mae', 'n_obs']]

results_df['oos_r2'] = results_df['oos_r2'] * 100  # convert to %
results_df.columns = ['OOS R² (%)', 'Hit Rate', 'Pearson r', 'Spearman r', 'MAE', 'N obs']
print(results_df.round(4).to_string())

## 4. Diebold-Mariano Test

In [ ]:
common = pred_sig.merge(pred_bench, on=['date', 'ticker'], suffixes=('_sig', '_bench'))
if len(common) > 10:
    e_sig   = common['realized_sig'].values   - common['predicted_sig'].values
    e_bench = common['realized_bench'].values - common['predicted_bench'].values
    dm = diebold_mariano_test(e_sig, e_bench, loss='squared')
    print(f'Diebold-Mariano test (signature vs benchmark):')
    print(f'  DM statistic:   {dm["statistic"]:.4f}')
    print(f'  p-value:        {dm["p_value"]:.4f}')
    print(f'  Mean loss diff: {dm["mean_loss_diff"]:.8f}')
    print(f'  Preferred:      Model {dm["preferred_model"]}')
else:
    print('Insufficient overlapping predictions for DM test.')

## 5. Fama-MacBeth Regression

In [ ]:
sig_feat_cols = [c for c in sig_features.columns if c not in ('date', 'ticker')]
# Use top 5 features (by variance) for readability
top5 = sig_feat_cols[:5]

merged_fm = sig_features.merge(
    panel[['date', 'ticker', 'ret_forward']].dropna(),
    on=['date', 'ticker'], how='inner'
).dropna(subset=top5 + ['ret_forward'])

try:
    fm = fama_macbeth_regression(merged_fm, top5, return_col='ret_forward')
    print(f'Fama-MacBeth Regression ({fm["n_dates"]} dates, avg {fm["avg_n_stocks"]:.0f} stocks/date)')
    print(f'\n{"Feature":<30} {"Coef":>10} {"t-stat":>10} {"p-value":>10}')
    print('-' * 65)
    for feat in top5:
        coef  = fm['coefficients'][feat]
        tstat = fm['t_statistics'][feat]
        pval  = fm['p_values'][feat]
        sig_flag = '***' if pval < 0.01 else ('**' if pval < 0.05 else ('*' if pval < 0.10 else ''))
        print(f'{feat[:30]:<30} {coef:>10.4f} {tstat:>10.3f} {pval:>10.4f} {sig_flag}')
except Exception as e:
    print(f'FM regression failed: {e}')

## 6. Portfolio Sort Analysis

In [ ]:
# Sort on the momentum feature (level-1 signature = cumulative return)
mom_col = top5[0]  # first signature feature proxies for momentum

merged_ps = sig_features[['date', 'ticker', mom_col]].merge(
    panel[['date', 'ticker', 'ret_forward']].dropna(),
    on=['date', 'ticker'], how='inner'
).dropna(subset=[mom_col, 'ret_forward'])

try:
    ps = portfolio_sort_analysis(
        features=merged_ps[['date', mom_col]].rename(columns={mom_col: 'signal'}),
        returns=merged_ps['ret_forward'],
        n_quantiles=5,
    )
    
    q_means = ps['quantile_returns'].mean()
    print(f'Portfolio Sort on Signature Feature (5 quantiles)')
    print(f'  Long-short mean return: {ps["mean_ls_ew"]*100:.4f}%')
    print(f'  Newey-West t-stat:      {ps["t_stat_ew"]:.3f}')
    print(f'  p-value:                {ps["p_value_ew"]:.4f}')
    print(f'  Annualised Sharpe:      {ps["sharpe_ew"]:.3f}')
    
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(range(1, 6), q_means.values * 100, color=['red']*2 + ['grey'] + ['green']*2,
           alpha=0.8, edgecolor='k')
    ax.set_xticks(range(1, 6))
    ax.set_xticklabels([f'Q{i}' for i in range(1, 6)])
    ax.set_xlabel('Quintile (Q1=bottom, Q5=top signal)')
    ax.set_ylabel('Mean Return (%)')
    ax.set_title(f'Portfolio Sort Returns\nL/S={ps["mean_ls_ew"]*100:.4f}%, t={ps["t_stat_ew"]:.2f}')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'Portfolio sort failed: {e}')

In [ ]:
# Summary of all results
print('\n' + '='*60)
print('EMPIRICAL RESULTS SUMMARY')
print('='*60)
print(f'Signature OOS R²:   {sum_sig["oos_r2"]*100:.4f}%')
print(f'Benchmark OOS R²:   {sum_bench["oos_r2"]*100:.4f}%')
print(f'Signature hit rate: {sum_sig["hit_rate"]:.4f}')
print(f'Benchmark hit rate: {sum_bench["hit_rate"]:.4f}')
print('\nNote: OOS R² in cross-sectional equity prediction is typically')
print('very small (0.1%–2%) but economically meaningful.')